<a href="https://colab.research.google.com/github/charlesanthony1996/explainable-ai-assignment1/blob/main/lab_track_a_backup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1 - load the uci dataset

In [1]:
!pip install lime numpy pandas torch

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283913 sha256=573c4e4359b6bd135901500c28f48c535528741c48c8eeb08f24423c066a66b8
  Stored in directory: /root/.cache/pip/wheels/7c/04/5c/157dc9106512a6c7a30653ec064490c94a49e0fc8f63d19ab9
Successfully built lime


In [2]:
import os
import glob
import copy
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

from lime.lime_tabular import LimeTabularExplainer

In [3]:
 # to show wide tables in the terminal
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

seed = 42

np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
data_url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
zip_path = "online_retail_ii.zip"

if len(glob.glob("*.xlsx")) == 0:
  print("downloading the dataset...")

  urllib.request.urlretrieve(data_url, zip_path)

  with zipfile.ZipFile(zip_path) as zip_file:
    zip_file.extractall(".")

downloading the dataset...


In [5]:
# find the excel file that came out of the zip
excel_files = glob.glob("*.xlsx")
print(excel_files)

excel_path = excel_files[0]

['online_retail_II.xlsx']


In [ ]:
# the excel file has two sheets
sheets = pd.read_excel(excel_path, sheet_name=None)
sheet_names = list(sheets.keys())

print(sheet_names)

first_sheet = sheets[sheet_names[0]]
second_sheet = sheets[sheet_names[1]]

In [ ]:
# remove the overlapping days from the first sheet
second_sheet_start = second_sheet["InvoiceDate"].min()

first_sheet = first_sheet[first_sheet["InvoiceDate"] < second_sheet_start]

raw = pd.concat(
    [first_sheet, second_sheet],
    ignore_index=True
)

print("rows: ", raw.shape[0])
print("From: ", raw["InvoiceDate"].min())
print("To: ", raw["InvoiceDate"].max())


print(raw.head())

2. clean the data

keep only real products.
product codes start with 5 digits.
remove rows with price 0 or lower
keep cancellations for now, since they have negative quantites. this affects our net quantity sold per week

In [ ]:
data = raw.copy()

data["Invoice"] = data["Invoice"].astype(str)
data["StockCode"] = data["StockCode"].astype(str)

# mark cancelled invoices
data["is_cancellation"] = data["Invoice"].str.startswith("C")

# keep only real products
is_product = data["StockCode"].str.match(r"^\d{5}")
data = data[is_product]

# remove zero or negative prices
data = data[data["Price"] > 0]

print("rows after cleaning: ", data.shape[0])
print("cancelled rows: ", data["is_cancellation"].sum())

In [ ]:
# day of the transation (time removed)
data["date"] = data["InvoiceDate"].dt.normalize()

# monday of the week the transaction belongs to
days_since_monday = pd.to_timedelta(data["date"].dt.weekday, unit="D")
# print(days_since_monday)
data["week"] = data["date"] - days_since_monday

print(data[["InvoiceDate", "date", "week"]].head())

3. weekly aggregation and input features

we aggregate by week.
daily sales per product are very sparse

the dataset has no promotion or stock columns, so we build proxies


In [ ]:
# shop availability : open days per week

In [ ]:
open_days = data[["date", "week"]].drop_duplicates()
# print(open_days)

days_open_per_week = open_days.groupby("week")["date"].count()
# print(days_open_per_week)
days_open_per_week = days_open_per_week.rename("days_open")
# print(days_open_per_week)
days_open_per_week = days_open_per_week.reset_index()
# print(days_open_per_week)

print(days_open_per_week.head())
print(days_open_per_week.tail())

In [ ]:
# net quantity per product per week
# sum over all rows, including cancellations (negative quantities here)

weekly_quantity = data.groupby(["StockCode", "week"])["Quantity"].sum()
# print(weekly_quantity.head())

weekly_quantity = weekly_quantity.rename("net_quantity")
# print(weekly_quantity.head())

weekly_quantity = weekly_quantity.reset_index()

print(weekly_quantity.head())

In [ ]:
# price and days on sale per product per week

# only real sales (no cancellations).
# the weekly price is the quantity weighted average price : revenue / units

is_sale = (data["is_cancellation"] == False) & (data["Quantity"] > 0)

sales = data[is_sale].copy()

sales["revenue"] = sales["Quantity"] * sales["Price"]

weekly_sales = sales.groupby(["StockCode", "week"]).agg(
    sold_quantity=("Quantity", "sum"),
    revenue=("revenue", "sum"),
    days_on_sale=("date", "nunique")
)

weekly_sales = weekly_sales.reset_index()
print(weekly_sales.head())

weekly_sales["price"] = weekly_sales["revenue"] / weekly_sales["sold_quantity"]

print(weekly_sales.head())

In [ ]:
# time split and product selection

# we split by time, never randomly, because we forecast the future

validation_start = pd.Timestamp("2011-04-04")
test_start = pd.Timestamp("2011-07-04")

n_products = 30

train_sales = weekly_sales[weekly_sales["week"] < validation_start]

weeks_per_product = train_sales.groupby("StockCode")["week"].count()
weeks_per_product = weeks_per_product.sort_values(ascending=False)

selected_products = list(weeks_per_product.index[:n_products])
print(len(selected_products))

# product names, for pricing
descriptions = data.drop_duplicates("StockCode")
descriptions = descriptions.set_index("StockCode")["Description"]

for product in selected_products[:10]:
  print(product, "-", descriptions[product], "-", weeks_per_product[product], "weeks")

In [ ]:
# build one row for every product and every week

# weeks with no sales must still exist (but the quantity will be 0)

# the first and last week of the dataset are incomplete, so we drop them

first_week = data["week"].min() + pd.Timedelta(weeks=1)
print(first_week)

last_week = data["week"].max() - pd.Timedelta(weeks=1)
print(last_week)

all_weeks = pd.date_range(first_week, last_week, freq="7D")

print("weeks: ", len(all_weeks))
print("from: ", all_weeks[0].date(), "to", all_weeks[-1].date())

panel_parts = []

for product in selected_products:

  product_frame = pd.DataFrame({"week": all_weeks})
  product_frame["StockCode"] = product

  panel_parts.append(product_frame)

panel = pd.concat(panel_parts, ignore_index=True)

# print(panel.shape)

# add net quantity
panel = panel.merge(
    weekly_quantity,
    on=["StockCode", "week"],
    how="left"
)

# add price and days on sale
panel = panel.merge(
    weekly_sales[["StockCode", "week", "price", "days_on_sale"]],
    on=["StockCode", "week"],
    how="left"
)

# add shop open days
panel = panel.merge(
    days_open_per_week,
    on="week",
    how="left"
)

print(panel.head())

In [ ]:
# fill the gaps

# no row -> nothing sold -> quantity 0, days on sale 0
# no row in the shop calendar -> shop closed -> days open 0
# no prce in a week without sales -> use the last know price of that product

panel["quantity"] = panel["net_quantity"].fillna(0)
panel["quantity"] = panel["quantity"].clip(lower=0)

panel["days_on_sale"] = panel["days_on_sale"].fillna(0)
panel["days_open"] = panel["days_open"].fillna(0)

panel = panel.sort_values(["StockCode", "week"])
panel = panel.reset_index(drop=True)

# print(panel.head())

# last known price
panel["price"] = panel.groupby("StockCode")["price"].ffill()
panel["price"] = panel.groupby("StockCode")["price"].bfill()


# usual price = median weekly price of the product? why median?
# training period only (so no information from the test period)

train_panel = panel[panel["week"] < validation_start]

usual_price = train_panel.groupby("StockCode")["price"].median()
# print(usual_price.head())

usual_price = usual_price.rename("usual_price")
usual_price = usual_price.reset_index()

panel = panel.merge(
    usual_price,
    on="StockCode",
    how="left"
)

panel["price_ratio"] = panel["price"] / panel["usual_price"]

panel["promotion"] = (panel["price_ratio"] < 0.9).astype(int)

panel["month"] = panel["week"].dt.month

features = [
    "quantity",
    "price_ratio",
    "promotion",
    "days_on_sale",
    "days_open",
    "month"
]

print(panel[["StockCode", "week"] + features].head(10))

# look at one product over time
example_rows = panel[panel["StockCode"] == selected_products[0]]

plt.figure(figsize=(12, 4))

plt.plot(example_rows["week"], example_rows["quantity"])

plt.axvline(validation_start, color="orange", linestyle="--", label="validation start")
plt.axvline(test_start, color="red", linestyle="--", label="test start")

plt.title("Weekly quantity: " + str(descriptions[selected_products[0]]))
plt.ylabel("units per week")
plt.legend()

plt.show()

4. time windows

each sample has:-

input - the last 8 weeks x 6 features -> shape(8, 6)
target - quantity in the following week

example - weeks 1-8 -> predict week 9, weeks 2-9 -> predict week 10

In [ ]:
# window = 8

window_size = 8

window_list = []
target_list = []
product_list = []
target_week_list = []

for product in selected_products:

  product_rows = panel[panel["StockCode"] == product]
  product_rows = product_rows.sort_values("week")

  feature_values = product_rows[features].to_numpy(dtype=float)
  print(feature_values)
  quantities = product_rows["quantity"].to_numpy(dtype=float)

  weeks = product_rows["week"].to_numpy()

  for end in range(window_size, len(product_rows)):

    window = feature_values[end - window_size : end]
    target = quantities[end]
    target_week = weeks[end]

    window_list.append(window)
    target_list.append(target)
    product_list.append(product)

    target_week_list.append(target_week)


x_all = np.array(window_list)
print(x_all)
y_all = np.array(target_list)

products_all = np.array(product_list)

target_weeks_all = pd.to_datetime(np.array(target_week_list))

print("x shape: ", x_all.shape)
print("y shape: ", y_all.shape)

is_train = target_weeks_all < validation_start
is_validation = (target_weeks_all >= validation_start) & (target_weeks_all < test_start)
is_test = target_weeks_all >= test_start

x_train_raw = x_all[is_train]
y_train_raw = y_all[is_train]

x_val_raw = x_all[is_validation]
y_val_raw = y_all[is_validation]

x_test_raw = x_all[is_test]
y_test_raw = y_all[is_test]

products_test = products_all[is_test]
target_weeks_test = target_weeks_all[is_test]

print("train: ", x_train_raw.shape)
print("validation: ", x_val_raw.shape)
print("test: ", x_test_raw.shape)





In [ ]:
# scaling

# raw windows stay in human readable units
# here those units are units, price ratio , 0/1 , days, month
# lime will work on these values so its explanations are easy to read.

In [ ]:
quantity_column = features.index("quantity")

def log_quantity(windows):

  transformed = windows.copy()

  quantities = transformed[:, :, quantity_column]
  quantities = np.clip(quantities, 0, None)

  transformed[:, :, quantity_column] = np.log1p(quantities)

  return transformed

# mean and standard deviation per feature, from the tracking windows
train_transformed = log_quantity(x_train_raw)

feature_mean = train_transformed.mean(axis=(0, 1))
feature_std = train_transformed.std(axis=(0, 1))

# avoid division by zero for a current feature
feature_std[feature_std == 0] = 1.0

In [ ]:
def prepare_for_lstm(windows):

  transformed = log_quantity(windows)

  standardized = (transformed - feature_mean) / feature_std

  return standardized.astype(np.float32)


x_train = prepare_for_lstm(x_train_raw)
x_val = prepare_for_lstm(x_val_raw)
x_test = prepare_for_lstm(x_test_raw)

# the lstm predicts log(1 + quantity)
y_train = np.log1p(y_train_raw).astype(np.float32)
y_val = np.log1p(y_val_raw).astype(np.float32)
y_test = np.log1p(y_test_raw).astype(np.float32)

scaling_table = pd.DataFrame({
    "feature": features,
    "mean": feature_mean,
    "std": feature_std
})

print(scaling_table)

In [ ]:
# 5. train a small lstm

# 1 lstm layer with 32 hidden units
# the hidden state of the last week goes into a linear layer -> 1 number
# loss = mean squared error on log(1+ quantity)
# we keep the model from the epoch with the lowest validation loss

In [ ]:
class SalesLSTM(nn.Module):

  def __init__(self, n_features, hidden_size=32):

    super().__init__()

    self.lstm = nn.LSTM(
        input_size=n_features,
        hidden_size=hidden_size,
        batch_first=True
    )

    self.output_layer = nn.Linear(hidden_size, 1)

  def forward(self, x):

    lstm_output, _ = self.lstm(x)

    last_week = lstm_output[:, -1, :]

    prediction = self.output_layer(last_week)

    prediction = prediction.squeeze(-1)

    return prediction


train_dataset = TensorDataset(
    torch.tensor(x_train),
    torch.tensor(y_train)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

x_val_tensor = torch.tensor(x_val).to(device)
y_val_tensor = torch.tensor(y_val).to(device)

torch.manual_seed(seed)

model = SalesLSTM(n_features=len(features))
model = model.to(device)

loss_function = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

n_epochs = 200

train_losses = []
val_losses = []

best_val_loss = float("inf")
best_state = None
best_epoch = 0

for epoch in range(n_epochs):

  model.train()

  epoch_loss = 0.0

  for batch_x, batch_y in train_loader:

    batch_x = batch_x.to(device)
    batch_y = batch_y.to(device)

    optimizer.zero_grad()

    prediction = model(batch_x)
    loss = loss_function(prediction, batch_y)

    loss.backward()
    optimizer.step()

    epoch_loss = epoch_loss + loss.item() * len(batch_x)

  epoch_loss = epoch_loss / len(train_dataset)

  # validation
  model.eval()

  with torch.no_grad():
    val_prediction = model(x_val_tensor)
    val_loss = loss_function(val_prediction, y_val_tensor).item()

  train_losses.append(epoch_loss)
  val_losses.append(val_loss)

  # remember the best model
  if val_loss < best_val_loss:
    best_val_loss = val_loss
    best_state = copy.deepcopy(model.state_dict())
    best_epoch = epoch + 1

  if (epoch + 1) % 10 == 0:
    print(f"Epoch {epoch + 1:3d}  train loss {epoch_loss:.4f} val_loss {val_loss:.4f}")

model.load_state_dict(best_state)

print("Best epoch: ", best_epoch, "with validation loss", round(best_val_loss, 4))

plt.figure(figsize=(8, 4))

plt.plot(train_losses, label="train")
plt.plot(val_losses, label="validation")

plt.axvline(best_epoch - 1, color="gray", linestyle="--", label="best epoch")

plt.xlabel("Epoch")
plt.ylabel("MSE on log(1+ quantity)")
plt.title("LSTM training")
plt.show()



In [ ]:
# 5.1 prediction function in units

# this function takes raw windows (n, 8, 6) prepares them
# runs the lstm
# converts it back to units
# quantity = exp(prediction) - 1

def lstm_predict_quantity(raw_windows):

  model_input = prepare_for_lstm(raw_windows)

  input_tensor = torch.tensor(model_input).to(device)

  model.eval()

  with torch.no_grad():
    log_prediction = model(input_tensor)

  log_prediction = log_prediction.cpu().numpy()

  quantity = np.expm1(log_prediction)
  quantity = np.clip(quantity, 0, None)


  return quantity

In [ ]:
# 6 compare with baselines

# naive: next week = this week
# linear: ridge regression on the flattened, prepared window

# metrics on the test set, in units

# mae - average absolute error in units
# rmse - like mae, but large errors count more

# naive baseline: last observed quantity
naive_prediction = x_test_raw[:, -1, quantity_column]

# linear baseline: flatten (n, 8, 6) -> (n, 48)
n_train = x_train.shape[0]
n_test = x_test.shape[0]
n_flat = window_size * len(features)

x_train_linear = x_train.reshape(n_train, n_flat)
x_test_linear = x_test.reshape(n_test, n_flat)

linear_model = Ridge(alpha=1.0)

linear_model.fit(x_train_linear, y_train)

linear_log_prediction = linear_model.predict(x_test_linear)

linear_prediction = np.expm1(linear_log_prediction)
linear_prediction = np.clip(linear_prediction, 0, None)

lstm_prediction = lstm_predict_quantity(x_test_raw)

def evaluate(name, actual, predicted):

  mae = mean_absolute_error(actual, predicted)
  rmse = np.sqrt(mean_squared_error(actual, predicted))

  return {
      "model": name,
      "MAE": round(mae, 1),
      "RMSE": round(rmse, 1)
  }

results = []

results.append(evaluate("Naive (last week)", y_test_raw, naive_prediction))
results.append(evaluate("Linear (Ridge)", y_test_raw, linear_prediction))
results.append(evaluate("LSTM", y_test_raw, lstm_prediction))


results_table = pd.DataFrame(results)

print(results_table)

# actual vs predicted for one product in the test period
plot_product = selected_products[0]

is_plot_product = products_test == plot_product

plt.figure(figsize=(12, 4))

plt.plot(target_weeks_test[is_plot_product], y_test_raw[is_plot_product], label="actual", linewidth=2)
plt.plot(target_weeks_test[is_plot_product], naive_prediction[is_plot_product], label="naive")
plt.plot(target_weeks_test[is_plot_product], linear_prediction[is_plot_product], label="linear")
plt.plot(target_weeks_test[is_plot_product], lstm_prediction[is_plot_product], label="lstm")

plt.title("Test period: " + str(descriptions[plot_product]))
plt.ylabel("Units per week")
plt.legend()

plt.show()



In [ ]:
# 7 lime explanation for one held out product week

n_train_raw = x_train.shape[0]

x_train_flat = x_train_raw.reshape(n_train_raw, n_flat)

# feature names in the same order as the flattening
feature_names = []

for step in range(window_size):

  lag = window - step

  for feature in features:

    feature_names.append(f"{feature}_t - {lag}")

print(feature_names[:6])

print("...")

print(feature_names[:6])

print("...")

print(feature_names[-6:])


# promotion is either 0 or 1 and month(1-12) are categories, not continuous numbers
categorical_indices = []

for index, name in enumerate(feature_names):

  if name.startswith("promotion") or name.startswith("month"):

    categorical_indices.append(index)

print("Categorical features: ", len(categorical_indices))


def predict_for_lime(flat_rows):

  n_rows = flat_rows.shape[0]

  # reshape the perturbations back into windows
  windows = flat_rows.reshape(n_rows, window, len(features))

  predictions = lstm_predict_quantity(windows)

  return predictions

# check: flattening + reshaping must give the same predictions

